## Leaf Classification

This project classifies leaf species from pre-extracted numeric features (margin, shape, and texture descriptors), rather than from the raw leaf images themselves. It is a 99-class classification problem with very few examples per class, which makes it a good exercise in careful validation and calibrated probabilities.

## Approach
1. Load the data and explore the species balance and feature scale
2. Encode species labels and scale the numeric features
3. Train a baseline classifier and evaluate log loss on a validation split
4. Generate class probabilities and create the submission file
5. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c leaf-classification
!unzip -oq leaf-classification.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.7 MB/s eta 0:00:00
100% 34.3M/34.3M [00:00<00:00, 65.1MB/s]



In [3]:
!unzip -oq train.csv.zip
!unzip -oq test.csv.zip
!ls

images.zip		 sample_data		    test.csv	  train.csv
leaf-classification.zip  sample_submission.csv.zip  test.csv.zip  train.csv.zip


In [4]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Columns: 194 entries, id to texture64
dtypes: float64(192), int64(1), object(1)
memory usage: 1.5+ MB


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594 entries, 0 to 593
Columns: 193 entries, id to texture64
dtypes: float64(192), int64(1)
memory usage: 895.8 KB


In [7]:
train.sample(1)

,id,species,margin1,margin2,margin3,margin4,margin5,margin6,margin7,margin8,...,texture55,texture56,texture57,texture58,texture59,texture60,texture61,texture62,texture63,texture64
395,649,Tilia_Tomentosa,0.003906,0.0,0.013672,0.017578,0.03125,0.013672,0.021484,0.005859,...,0.0,0.005859,0.0,0.001953,0.03125,0.0,0.0,0.019531,0.0,0.039062


In [8]:
test.sample(1)

,id,margin1,margin2,margin3,margin4,margin5,margin6,margin7,margin8,margin9,...,texture55,texture56,texture57,texture58,texture59,texture60,texture61,texture62,texture63,texture64
12,39,0.013672,0.0,0.046875,0.019531,0.005859,0.0,0.041016,0.0,0.013672,...,0.0,0.0,0.00293,0.0,0.026367,0.0,0.0,0.0,0.0,0.018555


In [9]:
print(train.shape, test.shape)

(990, 194) (594, 193)


In [10]:
train["species"].nunique()

99

In [11]:
train.columns[:8].tolist()

['id',
 'species',
 'margin1',
 'margin2',
 'margin3',
 'margin4',
 'margin5',
 'margin6']

In [12]:
from sklearn.preprocessing import LabelEncoder

test_ids = test["id"]

le = LabelEncoder()
y = le.fit_transform(train["species"])
x = train.drop(columns=["id", "species"])
x_test = test.drop(columns=["id"])

In [14]:
x.shape, y.shape, x_test.shape

((990, 192), (990,), (594, 192))

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

model = LogisticRegression(max_iter=2000, C=10)
model.fit(x_train, y_train)

proba = model.predict_proba(x_val)
print("Validation log loss:", round(log_loss(y_val, proba, labels=range(99)), 4))

Validation log loss: 0.0572


In [17]:
x = scaler.fit_transform(x)
x_test = scaler.transform(x_test)

model.fit(x, y)

proba_test = model.predict_proba(x_test)

submission = pd.DataFrame(proba_test, columns=le.classes_)
submission.insert(0, "id", test_ids)
submission.to_csv("submission.csv", index=False)

In [18]:
!kaggle competitions submit -c leaf-classification -f submission.csv -m "Scaled LogisticRegression baseline"

100% 1.25M/1.25M [00:00<00:00, 1.95MB/s]
99 submissions remaining today.
Successfully submitted to Leaf Classification

In [21]:
cols = [c for c in train.columns if c not in ["id", "species"]]

sample_leaves = train[["species"] + cols].drop_duplicates("species")

meta = {"classes": list(le.classes_), "feature_cols": cols, "sample_leaves": sample_leaves}

import pickle, os
pickle.dump(model, open("leaf_model.pkl", "wb"))
pickle.dump(scaler, open("leaf_scaler.pkl", "wb"))
pickle.dump(meta, open("leaf_meta.pkl", "wb"))
print("sizes KB:", [os.path.getsize(f) // 1024 for f in ["leaf_model.pkl", "leaf_scaler.pkl", "leaf_meta.pkl"]])

sizes KB: [150, 7, 154]
